In [ ]:
import os
import numpy as np
from fipy import *
from tqdm import tqdm  
import cupy as cp
import cupyx.scipy.ndimage as cnd
cp.cuda.Device(0).use()
DTYPE = cp.float64

# ---------- GPU kernel ----------
def _gpu_kernels(dx, dy, dtype=DTYPE):
    kx  = cp.asarray([[0,0,0],[-1,0,1],[0,0,0]], dtype=dtype) / (2*dx)
    ky  = cp.asarray([[0,-1,0],[0,0,0],[0,1,0]], dtype=dtype) / (2*dy)
    kxx = cp.asarray([[0,0,0],[1,-2,1],[0,0,0]], dtype=dtype) / (dx*dx)
    kyy = cp.asarray([[0,1,0],[0,-2,0],[0,1,0]], dtype=dtype) / (dy*dy)
    return kx, ky, (kxx + kyy)

# ---------- GPU time stepping (explicit Euler, Neumann 0) ----------
def gpu_solve_diffusion(u0_np, nu, dt, steps, dx, dy):
    kx, ky, klap = _gpu_kernels(dx, dy)
    u = cp.asarray(u0_np, dtype=DTYPE)
    lap = cp.empty_like(u)
    for _ in range(steps):
        cnd.correlate(u, klap, mode='nearest', output=lap)
        u = u + dt * (nu * lap)
    return cp.asnumpy(u)

def gpu_solve_advection(u0_np, vel, dt, steps, dx, dy):
    kx, ky, _ = _gpu_kernels(dx, dy)
    u = cp.asarray(u0_np, dtype=DTYPE)
    dudx = cp.empty_like(u); dudy = cp.empty_like(u)
    for _ in range(steps):
        cnd.correlate(u, kx, mode='nearest', output=dudx)
        cnd.correlate(u, ky, mode='nearest', output=dudy)
        u = u - dt * vel * (dudx + dudy)
    return cp.asnumpy(u)

def gpu_solve_advdiff(u0_np, vel, nu, dt, steps, dx, dy):
    kx, ky, klap = _gpu_kernels(dx, dy)
    u = cp.asarray(u0_np, dtype=DTYPE)
    dudx = cp.empty_like(u); dudy = cp.empty_like(u); lap = cp.empty_like(u)
    for _ in range(steps):
        cnd.correlate(u, kx,   mode='nearest', output=dudx)
        cnd.correlate(u, ky,   mode='nearest', output=dudy)
        cnd.correlate(u, klap, mode='nearest', output=lap)
        u = u - dt * vel * (dudx + dudy) + dt * (nu * lap)
    return cp.asnumpy(u)

# Directories
train_folder = "train_data"
test_folder = "test_data"
os.makedirs(train_folder, exist_ok=True)
os.makedirs(test_folder, exist_ok=True)

# Set up the 2D domain and mesh
nx = 64
ny = 64
Lx = 1
Ly = 1
dx = Lx / nx
dy = Ly / ny
mesh = Grid2D(dx=dx, dy=dy, nx=nx, ny=ny)
x, y = mesh.cellCenters[0], mesh.cellCenters[1]

# Simulation time parameters optimized by Bongseok
timeStep = 0.0001
steps = 500

# Number of datasets to generate
num_datasets = 100000 # 70000

# Loop with tqdm progress bar
for i in tqdm(range(num_datasets), desc="Generating datasets"):


    # ----------- Advection-Diffusion Simulation ----------------##
    diffusivity_adv_diff = np.random.uniform(0.1, 0.4)
    velocity_x_adv_diff = 4
    velocity_y_adv_diff = 2
    vel_adv_diff = numerix.sqrt(velocity_x_adv_diff**2 + velocity_y_adv_diff**2)

    center_x = np.random.uniform(0.2 * Lx, 0.8 * Lx)
    center_y = np.random.uniform(0.2 * Ly, 0.8 * Ly)
    initial_width = np.random.uniform(0.025, 0.075)
    x, y = mesh.cellCenters
    init_val_adv_diff = np.exp(-(((x - center_x)**2 + (y - center_y)**2) / initial_width))
    taper = numerix.sin(numerix.pi*x) * numerix.sin(numerix.pi*y)
    init_val_adv_diff = init_val_adv_diff * taper

    # GPU calculation
    U_adv_diff = gpu_solve_advdiff(init_val_adv_diff.reshape(nx, ny),
                               float(vel_adv_diff), float(diffusivity_adv_diff), float(timeStep), steps, float(dx), float(dy))
    phi_adv_diff = CellVariable(name="phi_adv_diff", mesh=mesh, value=U_adv_diff.ravel())
    

    # ----------- Diffusion Only Simulation ----------------##
    diffusivity_diff = np.random.uniform(0.1, 0.4)

    center_x = np.random.uniform(0.2 * Lx, 0.8 * Lx)
    center_y = np.random.uniform(0.2 * Ly, 0.8 * Ly)
    initial_width = np.random.uniform(0.025, 0.075)
    x, y = mesh.cellCenters
    init_val_diff = np.exp(-(((x - center_x)**2 + (y - center_y)**2) / initial_width))
    taper = numerix.sin(numerix.pi*x) * numerix.sin(numerix.pi*y)
    init_val_diff = init_val_diff * taper
    # GPU calculation
    U_diff = gpu_solve_diffusion(init_val_diff.reshape(nx, ny),
                                 float(diffusivity_diff), float(timeStep), steps, float(dx), float(dy))
    phi_diff = CellVariable(name="phi_diff", mesh=mesh, value=U_diff.ravel())

    # ----------- Advection Only Simulation ----------------##
    velocity_x_adv = np.random.uniform(2, 5)
    velocity_y_adv = 2
    vel_adv = numerix.sqrt(velocity_x_adv**2 + velocity_y_adv**2)

    center_x = np.random.uniform(0.2 * Lx, 0.8 * Lx)
    center_y = np.random.uniform(0.2 * Ly, 0.8 * Ly)
    initial_width = np.random.uniform(0.025, 0.075)
    x, y = mesh.cellCenters
    init_val_adve = np.exp(-(((x - center_x)**2 + (y - center_y)**2) / initial_width))
    taper = numerix.sin(numerix.pi*x) * numerix.sin(numerix.pi*y)
    init_val_adve = init_val_adve * taper

    # GPU calculation
    U_adve = gpu_solve_advection(init_val_adve.reshape(nx, ny),
                                 float(vel_adv), float(timeStep), steps, float(dx), float(dy))
    phi_adv = CellVariable(name="phi_adv", mesh=mesh, value=U_adve.ravel())

    # ----------- save ----------------##
    save_folder = train_folder

    filename_diff = os.path.join(save_folder, f"dataset_2d_{i:05d}_diff.npz")
    filename_adve = os.path.join(save_folder, f"dataset_2d_{i:05d}_adve.npz")
    filename_adve_diff = os.path.join(save_folder, f"dataset_2d_{i:05d}_adve_diff.npz")

    size = 64*64
    # save diff dataset
    pattern = np.array([1, 0, 0]).reshape(-1, 1)
    one_hot = np.zeros((size, 1))
    one_hot[:len(one_hot) // 3 * 3] = np.tile(pattern, (size // 3, 1))
    np.savez(filename_diff,
             last_value=phi_diff.value,        
             init_value=init_val_diff,
             label=0,
             one_hot=one_hot,
             diffusivity=diffusivity_diff,
             velocity_x=velocity_x_adv_diff,
             velocity_y=velocity_y_adv_diff)

    # save adve dataset
    pattern = np.array([0, 1, 0]).reshape(-1, 1)
    one_hot = np.zeros((size, 1))
    one_hot[:len(one_hot) // 3 * 3] = np.tile(pattern, (size // 3, 1))
    np.savez(filename_adve,
             last_value=phi_adv.value,           
             init_value=init_val_adve,
             label=1,
             one_hot=one_hot,
            #  diffusivity=diffusivity_adv,
             velocity_x=velocity_x_adv,
             velocity_y=velocity_y_adv)

    # save adve_diff dataset
    pattern = np.array([0, 0, 1]).reshape(-1, 1)
    one_hot = np.zeros((size, 1))
    one_hot[:len(one_hot) // 3 * 3] = np.tile(pattern, (size // 3, 1))
    np.savez(filename_adve_diff,
             last_value=phi_adv_diff.value,      
             init_value=init_val_adv_diff,
             label=2,
             one_hot=one_hot,
             diffusivity=diffusivity_adv_diff,
             velocity_x=velocity_x_adv_diff,
             velocity_y=velocity_y_adv_diff)

    # print(one_hot.shape)
    del phi_adv_diff, phi_diff, phi_adv
    del init_val_adv_diff, init_val_diff, init_val_adve
    del filename_diff, filename_adve, filename_adve_diff
    del save_folder
    del i
    del x, y